In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

# Параметры MinIO / S3
MINIO_ENDPOINT = "http://127.0.0.1:9000"
WAREHOUSE = "s3://mydatalab/warehouse"   # базовый warehouse для Iceberg

# Адрес Iceberg REST, который поднимает Gravitino
ICEBERG_REST_URI = "http://127.0.0.1:8181/iceberg"  # подставь свой порт/путь

spark = (
    SparkSession.builder
    .appName("GravitinoIcebergDemo")
    # Включаем Iceberg-расширения для Spark
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    # Регистрируем каталог 'gt', который ходит в Gravitino Iceberg REST
    .config("spark.sql.catalog.gt", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.gt.type", "rest")
    .config("spark.sql.catalog.gt.uri", ICEBERG_REST_URI)
    # .config("spark.sql.catalog.gt.warehouse", WAREHOUSE)
    .config("spark.sql.catalog.gt.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    # Параметры доступа к MinIO для Iceberg S3FileIO
    .config("spark.sql.catalog.gt.s3.endpoint", MINIO_ENDPOINT)
    .config("spark.sql.catalog.gt.s3.path-style-access", "true")
    .config("spark.sql.catalog.gt.s3.access-key-id", access_key)
    .config("spark.sql.catalog.gt.s3.secret-access-key", secret_key)
    .config("spark.sql.catalog.gt.s3.region", "us-east-1")
    .getOrCreate()
)

# Данные, как в исходном примере
df = (
    spark.createDataFrame([("pavel", 1), ("mark", 2)], ["name", "value"])
    .withColumn("ts", F.current_timestamp())
)

# 1) создаём namespace в Iceberg, если его нет
spark.sql("CREATE NAMESPACE IF NOT EXISTS gt.demo")

# 2) создаём/перезаписываем таблицу gt.demo.spark_test в Iceberg
df.writeTo("gt.demo.spark_test").createOrReplace()

# 3) читаем таблицу обратно через Iceberg-каталог
df2 = spark.table("gt.demo.spark_test")
df2.show()


Py4JJavaError: An error occurred while calling o73.createOrReplace.
: org.apache.iceberg.exceptions.RESTException: Unable to process: Not Acceptable
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:250)
	at org.apache.iceberg.rest.ErrorHandlers$ViewErrorHandler.accept(ErrorHandlers.java:168)
	at org.apache.iceberg.rest.ErrorHandlers$ViewErrorHandler.accept(ErrorHandlers.java:152)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:240)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:336)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:297)
	at org.apache.iceberg.rest.BaseHTTPClient.head(BaseHTTPClient.java:45)
	at org.apache.iceberg.rest.RESTSessionCatalog.viewExists(RESTSessionCatalog.java:1112)
	at org.apache.iceberg.rest.RESTSessionCatalog$Builder.replaceTransaction(RESTSessionCatalog.java:830)
	at org.apache.iceberg.rest.RESTSessionCatalog$Builder.createOrReplaceTransaction(RESTSessionCatalog.java: